# FASTQ to h5ad Pipeline (kallisto | bustools)
This pipeline aligns raw single-cell RNA-seq FASTQ files to the mm10 mouse reference genome and generates `.h5ad` count matrices.

### Steps:
1. **Reference Generation:** Build the transcriptome index from the genome `.fa` and `.gtf`.
2. **Quantification:** Align reads and count UMIs to generate cell x gene matrices.


In [8]:
import os
import glob

# 1. Define Base Directories
base_dir = '/home/nakagawa/datasets'
fastq_dir = os.path.join(base_dir, 'SRR_cortex/SRR_cortex_scRNA')
genome_dir = os.path.join(base_dir, 'genome', 'mm10')
output_dir = os.path.join(base_dir, 'SRR_cortex/SRR_cortex_scRNA/processed_h5ad')

os.makedirs(output_dir, exist_ok=True)

# 2. Define Reference Files
fasta_path = os.path.join(genome_dir, 'Mus_musculus.GRCm38.dna.primary_assembly.fa')
gtf_path = os.path.join(genome_dir, 'Mus_musculus.GRCm38.84.gtf')

# 3. Define Index Output Paths
index_path = os.path.join(genome_dir, 'transcriptome.idx')
t2g_path = os.path.join(genome_dir, 'transcripts_to_genes.txt')
t_fasta_path = os.path.join(genome_dir, 'transcriptome.fa')

print("Directories and paths configured.")


Directories and paths configured.


In [9]:
%%bash

FASTQ_DIR="/home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA"
OUT_DIR="/home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/processed_h5ad"

INDEX="/home/nakagawa/datasets/genome/mm10/transcriptome.idx"
T2G="/home/nakagawa/datasets/genome/mm10/transcripts_to_genes.txt"

mkdir -p "$OUT_DIR"

for R2 in ${FASTQ_DIR}/*_2.fastq.gz; do

    SAMPLE=$(basename "$R2" _2.fastq.gz)

    echo "---------------------------------------------------"
    echo "Processing $SAMPLE..."

    SAMPLE_OUT="$OUT_DIR/$SAMPLE"

    R1="$FASTQ_DIR/${SAMPLE}_1.fastq.gz"

    # Skip if R1 missing
    if [[ ! -f "$R1" ]]; then
        echo "Missing R1 for $SAMPLE"
        continue
    fi

    # Remove old failed outputs if they exist
    rm -rf "$SAMPLE_OUT"

    echo "Trying 10xv3..."

    kb count \
        -i "$INDEX" \
        -g "$T2G" \
        -x 10xv3 \
        -o "$SAMPLE_OUT" \
        --h5ad \
        "$R1" "$R2"

    STATUS=$?

    # If 10xv3 failed, retry with 10xv2
    if [[ $STATUS -ne 0 ]]; then

        echo "10xv3 failed for $SAMPLE"
        echo "Retrying with 10xv2..."

        rm -rf "$SAMPLE_OUT"

        kb count \
            -i "$INDEX" \
            -g "$T2G" \
            -x 10xv2 \
            -o "$SAMPLE_OUT" \
            --h5ad \
            "$R1" "$R2"

        STATUS=$?
    fi

    if [[ $STATUS -eq 0 ]]; then
        echo "Finished $SAMPLE successfully"
    else
        echo "FAILED: $SAMPLE"
    fi

done

---------------------------------------------------
Processing SRR12082755...
Trying 10xv3...


[2026-05-23 16:15:57,628]    INFO [count] Using index /home/nakagawa/datasets/genome/mm10/transcriptome.idx to generate BUS file to /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/processed_h5ad/SRR12082755 from
[2026-05-23 16:15:57,628]    INFO [count]         /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/SRR12082755_1.fastq.gz
[2026-05-23 16:15:57,628]    INFO [count]         /home/nakagawa/datasets/SRR_cortex/SRR_cortex_scRNA/SRR12082755_2.fastq.gz
Traceback (most recent call last):
  File "/home/nakagawa/anaconda3/envs/scrna/bin/kb", line 6, in <module>
    sys.exit(main())
  File "/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/ngs_tools/logging.py", line 62, in inner
    return func(*args, **kwargs)
  File "/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/kb_python/main.py", line 1933, in main
    COMMAND_TO_FUNCTION[args.command](parser, args, temp_dir=temp_dir)
  File "/home/nakagawa/anaconda3/envs/scrna/lib/python3.10/site-packages/kb

Error while terminating subprocess (pid=2005579): 


TypeError: %d format: a real number is required, not NoneType